## Marine heatwave visualization

Interactive Plotly map of the NW Mediterranean bounding box, with a time slider. Sea surface temperature is shown on a blue intensity scale for normal conditions; cells flagged as part of a detected marine heatwave event switch to red, overriding the temperature-based color.

Starting with a small prototype (a couple of months) to validate the visual style, before scaling to the full 2016-2026 period. The sampling resolution for the full-period animation was iterated on (weekly, then biweekly, then every 10 days) to balance animation smoothness against GitHub's 100MB file size limit.

In [1]:
import duckdb
import pandas as pd
import plotly.graph_objects as go

con = duckdb.connect()

In [2]:
# Look at the raw temperature data structure again, refresh
# (analysed_sst below is raw Kelvin, unconverted -- structural check, not a final display)
raw_sample = con.execute("""
    SELECT * FROM '../data/processed/med_sst_2016_2026.parquet' 
    LIMIT 5
""").df()
print(raw_sample)

        time   latitude  longitude  analysed_sst  __index_level_0__
0 2016-01-01  40.507652   2.043520    288.579994                  0
1 2016-01-01  40.507652   2.093567    288.649994                  1
2 2016-01-01  40.507652   2.143612    288.699994                  2
3 2016-01-01  40.507652   2.193659    288.739994                  3
4 2016-01-01  40.507652   2.243704    288.749994                  4


In [3]:
# Look at the events table structure
events_sample = con.execute("""
    SELECT * FROM '../data/processed/mhw_events.parquet' 
    LIMIT 5
""").df()
print(events_sample)

    latitude  longitude  group_id event_start  event_end  duration_days
0  41.059387   4.946186         0  2016-01-01 2016-01-13             13
1  40.658123   6.547657         0  2016-01-01 2016-01-16             16
2  41.059387   5.096323         0  2016-01-01 2016-01-13             13
3  40.858757  11.652344         0  2016-01-01 2016-02-09             40
4  40.607967   6.147289         0  2016-01-01 2016-01-17             17


In [4]:
# Test the interval JOIN for a small period (Jan-Mar 2016), which contains the verified 46-day event.
# Flags each cell/day as 'Y' if it falls inside a detected heatwave event, 'N' otherwise.
check_months = con.execute("""
    SELECT 
        raw.time,
        raw.latitude,
        raw.longitude,
        raw.analysed_sst,
        CASE WHEN events.event_start IS NOT NULL THEN 'Y' ELSE 'N' END AS in_heatwave
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    LEFT JOIN '../data/processed/mhw_events.parquet' AS events
        ON raw.latitude = events.latitude 
        AND raw.longitude = events.longitude
        AND raw.time BETWEEN events.event_start AND events.event_end
    WHERE raw.time BETWEEN '2016-01-01' AND '2016-03-31'
""").df()
pd.set_option('display.float_format', '{:.2f}'.format)
# analysed_sst stats below are raw Kelvin, unconverted -- structural verification only
print(check_months.describe())

                                time   latitude  longitude  analysed_sst
count                        1086176 1086176.00 1086176.00    1086176.00
mean   2016-02-15 00:00:00.000000256      42.06       7.81        287.10
min              2016-01-01 00:00:00      40.51       2.04        279.83
25%              2016-01-23 00:00:00      41.16       5.20        286.62
50%              2016-02-15 00:00:00      41.91       7.75        287.10
75%              2016-03-09 00:00:00      42.87      10.15        287.62
max              2016-03-31 00:00:00      44.47      13.95        290.73
std                              NaN       1.04       3.07          0.94


In [5]:
# Check the balance of flagged vs unflagged rows: 'Y' should be a small minority,
# concentrated in the cells and dates covered by the verified Jan-Feb 2016 event.
print(check_months["in_heatwave"].value_counts())

in_heatwave
N    1041939
Y      44237
Name: count, dtype: int64


In [6]:
# How many distinct cells have at least one 'Y' day in this period?
n_cells_flagged = check_months[check_months["in_heatwave"] == "Y"][["latitude", "longitude"]].drop_duplicates().shape[0]
print(f"Distinct cells with at least one heatwave day: {n_cells_flagged}")

Distinct cells with at least one heatwave day: 1858


**Results**: 44,237 flagged rows across the test period, spread over 1,858 distinct cells, roughly 15% of the bounding box's sea cells. Initially this seemed high compared to the previously verified event (75 cells sharing the same Jan 1 - Feb 15 2016 window, per notebook 03), but it makes physical sense: marine heatwaves are not point phenomena, they cover contiguous patches of ocean at synoptic scale, so a real event naturally touches hundreds or thousands of neighboring cells, not an isolated one.

## First static frame: single day test

Before building the full animated slider, testing the visual style (blue intensity for normal SST, red override for heatwave cells) on a single day known to be inside the verified event.

In [7]:
# Extract a single day's data for the visual test
day_test = check_months[check_months["time"] == "2016-01-20"].copy()
print(day_test.shape)
print(day_test["in_heatwave"].value_counts())

(11936, 5)
in_heatwave
N    11349
Y      587
Name: count, dtype: int64


In [8]:
# Assign color: red if in_heatwave, otherwise a blue shade based on temperature
fig = go.Figure()

normal_cells = day_test[day_test["in_heatwave"] == "N"]
heatwave_cells = day_test[day_test["in_heatwave"] == "Y"]

fig.add_trace(go.Scattergeo(
    lon=normal_cells["longitude"],
    lat=normal_cells["latitude"],
    mode="markers",
    marker=dict(
        size=4,
        color=normal_cells["analysed_sst"] - 273.15,
        colorscale="Blues",
        showscale=True,
        colorbar=dict(title="SST (°C)")
    ),
    name="Normal"
))

fig.add_trace(go.Scattergeo(
    lon=heatwave_cells["longitude"],
    lat=heatwave_cells["latitude"],
    mode="markers",
    marker=dict(size=4, color="red"),
    name="Heatwave"
))

fig.update_geos(
    lonaxis_range=[2, 14],
    lataxis_range=[40.5, 44.5],
    showcoastlines=True,
    showland=True
)

fig.update_layout(title="SST and detected heatwave cells (2016-01-20)", height=600)
fig.show()

In [9]:
# Check how many unique days are in the test period, to know how many animation frames we'll need
n_days = check_months["time"].nunique()
print(n_days)

91


In [10]:
# Build one frame per day: same two-trace structure as static test,
# but recalculated for each day in the period
unique_days = sorted(check_months["time"].unique())

# Global SST (°C) range across all 91 days, so the Blues colorbar stays fixed
# across frames instead of rescaling to each day's own min/max.
sst_celsius_all = check_months["analysed_sst"] - 273.15
sst_cmin = sst_celsius_all.min()
sst_cmax = sst_celsius_all.max()

frames = []
for day in unique_days:
    day_data = check_months[check_months["time"] == day]
    normal = day_data[day_data["in_heatwave"] == "N"]
    heatwave = day_data[day_data["in_heatwave"] == "Y"]

    # When no cells are flagged this day, keep a single point far outside the
    # map's lon/lat range: the trace stays non-empty (so its legend entry
    # persists -- Plotly drops a legend swatch entirely for a trace with zero
    # data points, even with showlegend=True) while the point itself is
    # clipped from view since it falls outside lonaxis_range/lataxis_range.
    if len(heatwave) > 0:
        hw_lon, hw_lat = heatwave["longitude"], heatwave["latitude"]
    else:
        hw_lon, hw_lat = [0.0], [0.0]

    frame = go.Frame(
        data=[
            go.Scattergeo(lon=normal["longitude"], lat=normal["latitude"], mode="markers",
                          marker=dict(size=4, color=normal["analysed_sst"] - 273.15, colorscale="Blues",
                                      cmin=sst_cmin, cmax=sst_cmax),
                          name="Normal", showlegend=True),
            go.Scattergeo(lon=hw_lon, lat=hw_lat, mode="markers",
                          marker=dict(size=4, color="red"),
                          name="Heatwave", showlegend=True)
        ],
        name=str(day)[:10]
    )
    frames.append(frame)

print(len(frames))

91


In [11]:
# Build the initial figure (first day) and attach all 91 frames, with a slider to move between them
first_day_data = check_months[check_months["time"] == unique_days[0]]
normal_first = first_day_data[first_day_data["in_heatwave"] == "N"]
heatwave_first = first_day_data[first_day_data["in_heatwave"] == "Y"]

if len(heatwave_first) > 0:
    hw_lon_first, hw_lat_first = heatwave_first["longitude"], heatwave_first["latitude"]
else:
    hw_lon_first, hw_lat_first = [0.0], [0.0]

fig = go.Figure(
    data=[
        go.Scattergeo(
            lon=normal_first["longitude"], lat=normal_first["latitude"], mode="markers",
            marker=dict(
                size=4, 
                color=normal_first["analysed_sst"] - 273.15,
                colorscale="Blues", showscale=True, 
                cmin=sst_cmin, cmax=sst_cmax,
                colorbar=dict(
                    title=dict(text="SST (°C)", side="top"),
                    x=0.84, xanchor="left",
                    y=0.74, yanchor="middle",
                    len=0.42,
                    thickness=16,
                )
            ),
            name="Normal", showlegend=True
        ),
        go.Scattergeo(
            lon=hw_lon_first, lat=hw_lat_first, mode="markers",
            marker=dict(size=4, color="red"),
            name="Heatwave", showlegend=True
        )
    ],
    frames=frames
)

# Map fills the full plot height and 80% of the width; the remaining 20% on the
# right holds the colorbar and legend, stacked close together (see below).
fig.update_geos(
    lonaxis_range=[2, 14], lataxis_range=[40.5, 44.5], 
    showcoastlines=True, showland=True,
    domain=dict(x=[0, 0.80], y=[0, 1])
)

# Label every step with its full date and let Plotly's own overlap-avoidance
# pick the spacing: with this figure width/font it lands at ~6 days, close to
# weekly, without hiding any step's ability to be scrubbed to directly.
slider_steps = []
for f in frames:
    slider_steps.append({
        "args": [[f.name], {"frame": {"duration": 0, "redraw": True}}],
        "label": pd.to_datetime(f.name).strftime("%b %d"),
        "method": "animate"
    })

fig.update_layout(
    title=dict(
        text="SST and detected heatwave cells (Jan-Mar 2016)",
        x=0.5, xanchor="center",
        y=0.99, yanchor="top",
        font=dict(size=20)
    ),
    height=460,
    width=1150,
    margin=dict(t=55, b=90, l=10, r=10),
    legend=dict(
        x=0.84, xanchor="left",
        y=0.46, yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(0,0,0,0.2)", borderwidth=1
    ),
    sliders=[{
        "steps": slider_steps,
        "x": 0.0, "len": 0.80,
        "y": -0.10,
        "pad": {"t": 30, "b": 10},
        "currentvalue": {"prefix": "Date: ", "font": {"size": 13}},
        "font": {"size": 12}
    }]
)

fig.show()

## Scaling to the full 2016-2026 period

Same interval JOIN and heatwave flagging logic as the test period, now applied to the full dataset. The sampling resolution was iterated to balance animation smoothness against GitHub's 100MB file size limit: weekly sampling (one day every 7, 547 frames) exported at ~138MB, over the limit; biweekly sampling (one day every 14, 274 frames) brought that down to 72MB, with room to spare; 10-day sampling (383 frames) was settled on as the densest resolution that still fits, exporting at 98MB (see final results below).

**Note on spatial coverage**: the northeastern corner of the visualization (upper Adriatic) shows heatwave activity that was already included in the original 253,145-event count from notebook 03, not a new or separate finding. It's simply the first time this edge of the bounding box is visible in the animation, since the earlier test period (Jan-Mar 2016) happened not to show strong signal there. The total event count is unchanged.

## Aesthetic pass: neutral grey landmass

Testing a scientific-convention color scheme: neutral grey for the landmass (darker grey than the sea, to create clear contrast without competing with the data), a light grey coastline for a soft but visible boundary, and no separate ocean fill since the sea area is already covered by the temperature data itself. This follows standard practice in oceanographic figures, where land color should provide context without drawing attention away from the data.

**Note on coastline**: the plotted coastline (Plotly's built-in vector boundary) and the actual CMEMS sea/land mask are two independently-defined boundaries and don't perfectly align, small offsets are expected near the coast. This is a rendering limitation, not a data quality issue: the underlying CMEMS land/sea mask (verified in notebook 00) is unaffected.

To keep those offsets from reading as errors, `landcolor` and `bgcolor` are set to the same flat grey (`rgb(225, 225, 225)`) and the coastline is thinned and lightened (`coastlinewidth=1`, `coastlinecolor` close to the land tone instead of solid black). Compared against a few alternatives (solid black width-2 coastline; unified fill with a thin line; unified fill with the coastline removed entirely; a softer grey coastline over two distinct-but-close fills), the unified-fill-plus-thin-line version reads cleanest: since land and background are now the same color, a data cell landing just past the vector coastline (in either direction) blends into the surrounding tone instead of standing out as a grey-on-white or grey-on-black seam, while the thin line still gives a faint coastal reference. Dropping the coastline entirely looked equally clean but removes that reference entirely, and a thicker or darker line re-introduces the same hard edge the mismatch was showing through.

## Exporting the animation

Saving the final animated figure as a standalone HTML file, so it can be viewed with full interactivity outside the notebook (GitHub does not render interactive Plotly widgets in its notebook preview).

## Trying 10-day sampling to maximize frame density under the 100MB export limit

In [12]:
# Re-sample at every 10 days, targeting close to the 100MB GitHub limit
full_scale_query_10d = con.execute("""
    SELECT 
        raw.time, raw.latitude, raw.longitude, raw.analysed_sst,
        CASE WHEN events.event_start IS NOT NULL THEN 'Y' ELSE 'N' END AS in_heatwave
    FROM '../data/processed/med_sst_2016_2026.parquet' AS raw
    LEFT JOIN '../data/processed/mhw_events.parquet' AS events
        ON raw.latitude = events.latitude AND raw.longitude = events.longitude
        AND raw.time BETWEEN events.event_start AND events.event_end
    WHERE DATE_DIFF('day', '2016-01-01', raw.time) % 10 = 0
""").df()

print(full_scale_query_10d.shape)
print(full_scale_query_10d["time"].nunique())

(4571488, 5)
383


In [13]:
# Color scale range for the 10-day sampled data (final export resolution)
sst_cmin_10d = (full_scale_query_10d["analysed_sst"] - 273.15).min()
sst_cmax_10d = (full_scale_query_10d["analysed_sst"] - 273.15).max()
print(sst_cmin_10d, sst_cmax_10d)

4.859993787854933 30.479993215203308


In [14]:
# Build one frame per 10-day sampled day (final version used for the exported HTML)
unique_days_10d = sorted(full_scale_query_10d["time"].unique())

frames_10d = []
for day in unique_days_10d:
    day_data = full_scale_query_10d[full_scale_query_10d["time"] == day]
    normal = day_data[day_data["in_heatwave"] == "N"]
    heatwave = day_data[day_data["in_heatwave"] == "Y"]
    
    if len(heatwave) > 0:
        hw_lon, hw_lat = heatwave["longitude"], heatwave["latitude"]
    else:
        hw_lon, hw_lat = [0.0], [0.0]
    
    frame = go.Frame(
        data=[
            go.Scattergeo(lon=normal["longitude"], lat=normal["latitude"], mode="markers",
                          marker=dict(size=4, color=normal["analysed_sst"] - 273.15, 
                                      cmin=sst_cmin_10d, cmax=sst_cmax_10d, colorscale="Blues"),
                          name="Normal", showlegend=True),
            go.Scattergeo(lon=hw_lon, lat=hw_lat, mode="markers",
                          marker=dict(size=4, color="red"),
                          name="Heatwave", showlegend=True)
        ],
        name=str(day)[:10]
    )
    frames_10d.append(frame)

print(len(frames_10d))

383


In [15]:
# Final figure: 10-day resolution, densest sampling that fits under GitHub's 100MB limit
first_day_10d = full_scale_query_10d[full_scale_query_10d["time"] == unique_days_10d[0]]
normal_first_10d = first_day_10d[first_day_10d["in_heatwave"] == "N"]
heatwave_first_10d = first_day_10d[first_day_10d["in_heatwave"] == "Y"]

if len(heatwave_first_10d) > 0:
    hw_lon_first_10d, hw_lat_first_10d = heatwave_first_10d["longitude"], heatwave_first_10d["latitude"]
else:
    hw_lon_first_10d, hw_lat_first_10d = [0.0], [0.0]

fig_10d = go.Figure(
    data=[
        go.Scattergeo(
            lon=normal_first_10d["longitude"], lat=normal_first_10d["latitude"], mode="markers",
            marker=dict(
                size=4, 
                color=normal_first_10d["analysed_sst"] - 273.15,
                cmin=sst_cmin_10d, cmax=sst_cmax_10d,
                colorscale="Blues", showscale=True, 
                colorbar=dict(
                    title=dict(text="SST (°C)", side="top"),
                    x=0.84, xanchor="left",
                    y=0.74, yanchor="middle",
                    len=0.42,
                    thickness=16,
                )
            ),
            name="Normal", showlegend=True
        ),
        go.Scattergeo(
            lon=hw_lon_first_10d, lat=hw_lat_first_10d, mode="markers",
            marker=dict(size=4, color="red"),
            name="Heatwave", showlegend=True
        )
    ],
    frames=frames_10d
)

fig_10d.update_geos(
    lonaxis_range=[2, 14], lataxis_range=[40.5, 44.5], 
    showcoastlines=True, coastlinecolor="rgb(190, 190, 190)", coastlinewidth=1,
    showland=True, landcolor="rgb(225, 225, 225)",
    bgcolor="rgb(225, 225, 225)",
    domain=dict(x=[0, 0.80], y=[0, 1])
)

slider_steps_10d = [{
    "args": [[f.name], {"frame": {"duration": 0, "redraw": True}}],
    "label": pd.to_datetime(f.name).strftime("%b %Y"),
    "method": "animate"
} for f in frames_10d]

fig_10d.update_layout(
    title=dict(
        text="SST and detected heatwave cells (2016-2026, every 10 days)",
        x=0.5, xanchor="center",
        y=0.99, yanchor="top",
        font=dict(size=20)
    ),
    height=460,
    width=1150,
    margin=dict(t=55, b=90, l=30, r=10),
    legend=dict(
        x=0.84, xanchor="left",
        y=0.46, yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="rgba(0,0,0,0.2)", borderwidth=1
    ),
    sliders=[{
        "steps": slider_steps_10d,
        "x": 0.0, "len": 0.80,
        "y": -0.10,
        "pad": {"t": 30, "b": 10},
        "currentvalue": {"prefix": "Date: ", "font": {"size": 13}},
        "font": {"size": 12}
    }]
)

fig_10d.show()

In [16]:
fig_10d.write_html("../outputs/marine_heatwave_animation.html")

**Results**: settled on 10-day sampling (383 frames) as the densest resolution that fits under GitHub's 100MB file limit, exported at 98MB. Color scale range 4.86-30.48°C, marginally narrower than the weekly/biweekly versions (4.72-30.66°C), since 10-day sampling happens to miss the exact days of the absolute min/max by a few days, consistent with the same sampling-gap explanation noted earlier in this notebook.